# Gradient-descent baseline — independent-optimizer calibration (cardio-only, SI stack)

<details>
<summary>Answers the EFC reviewers' #1 ask with the **tightest** analog of EFC itself: an independent</summary>

**gradient descent** over the same 16 calibration parameters. EFC is *not* gradient descent -- it
forms no gradient field and minimises no scalar objective, which is exactly why it was renamed
from Embedded Gradient Descent for reviewer comment R2-m1; its cubic controllers drive each
parameter off its own per-target relative error from *inside* the integration. This notebook is
still the tightest baseline because it moves the *same* 16 parameters against the *same* target
errors, only from *outside* the model and through a real descent direction -- to check the targets
can be hit as well by an off-the-shelf optimizer and to reveal the
**identifiability** of the fit — is a converged point one of many on a manifold? (It is also the
concrete counterpart to why the MCMC siblings' walkers don't sit at the optimum: MCMC samples
posterior mass, gradient descent *descends* to the mode.)

Method: **batched Adam** (first-order momentum gradient descent) on the unit cube, with `nRestarts`
LHS multi-starts advanced **in lockstep** + a Laplace identifiability covariance. Every Adam step
evaluates the whole `nRestarts × (1 + 2P)` finite-difference stencil in **one** vmapped
`batchedCalibration` call, so the forward solve is batched exactly like emcee/cmaes/smc (per-point
scipy optimizers were `nRestarts × iterations` serial un-vmapped solves — multi-hour on CPU). The
gradient is central **finite differences** by default (`gradSource`); autodiff (`jax.grad` through
`modelClassSI.solveFinalArray`) is kept selectable but is fragile here — the model's
valve/beat/systole `jnp.where` kinks nan-poison the reverse-mode gradient, so FD (which only ever
evaluates the finite-guarded objective) is the robust default. Each objective eval is a
**controllers-off** forward sim at a fixed θ (`multiplierC = 0.0` single settle stage): θ is
scattered into the 16 param states of `Y0` and the targeted observables are read from the settled
final state (model-internal cycle-operator states, plus the algebraic `amp_P_As`).

**Why controllers-off calibration and not `mode="baseline"`:** `multiplierC=0` freezes only the
16 *cubic tuning knobs*; the `t_Sys_HC` **`polynomialController`** (systolic-timing-vs-heart-rate,
a *structural* physiological law, not a tuning knob) stays active. Genuine `mode="baseline"` drops
the whole calibration section, so `t_Sys_HC` freezes at the wrong default (0.4 vs ~0.56) and the
forward eval lands ~18% off the targets. The controllers-off calibration reproduces the targets to
~2% — the intrinsic cardiac-cycle phase-jitter floor — and is consistent with the convergence /
batch runs, which also drive `batchedCalibration`.

**Structure (like `Convergence_Run.ipynb`): Phase 1 runs the optimizer and writes an `.h5`;
Phase 2 loads that file and plots.** After a kernel restart, run the cheap setup cells + the
Phase-2 load cell to re-plot without re-fitting.

**Prior/box = the controller *numerical-safeguard* bounds** (`config/models/*.json`
`calibration[p].params.{minValue,maxValue}`), NOT the `convergence` LHS ranges. The latter are
initial-guess sampling ranges; the target-consistent region can fall outside them (e.g.
`E_Hl → ~19` vs LHS `[1,15]`), so a box on the LHS ranges would truncate it. The safeguards are
the hard bounds the calibration actually operates within, enforced by clipping each Adam step.

This notebook is **fully self-contained** — it seeds, fits, saves, and plots with **no EFC
dependency**. Cross-technique comparison against EFC lives in a separate comparison notebook.

Every knob lives in the `runConfig` dict below (project single-config-surface rule).

</details>

In [ ]:
# region -> runConfig (the ONE place run configuration lives; project single-config-surface rule)
####################################################################################################
# runConfig — the ONE place run configuration lives (project rule: see repo-root CLAUDE.md).
# Defined first so the device/precision block can be applied before JAX initialises below.
####################################################################################################
runConfig = {
    # --- file references ---------------------------------------------------
    "model":    "cvModel.json",   # config/models/ cardio-only model (16 calibration knobs + t_Sys_HC)
    "scenario": "sepsis.json",    # config/scenarios/ shared.twin + convergence.{parameters,observations}
    "mode":     "calibration",    # forward eval = a controllers-OFF calibration solve

    # --- device / precision (applied in the Imports cell, before `import jax`) ---------
    "device": {
        "useGpu":    False,       # CPU here (GPU is slower for this batched solve)
        "precision": "float64",   # "float64" (parity/reference) or "float32" (~faster on GPU)
    },

    # --- inference / fit backend (batched gradient descent over ONE forward evaluator + objective) ---
    "inference": {
        "method":      "gd",      # fixed: gradient-descent-only notebook (drives the output suffix)
        "run":         False,      # False -> plot-only: skip Phase 1 (fit); reload the saved .h5 and plot
        "whiten": {"enabled": False, "warmup": 512, "keep": 0.2},
        "seed":        0,         # init + backend reproducibility
        "init":        "lhs",     # restart init: "lhs" (screened LHS pool, see gd.initPool below)
        "settleRuns":  1,         # settle runs before reading obs
        "sigmaRel":    0.02,      # relative likelihood scale: sigma_i = sigmaRel * |target_i|
        "chunkSize":   1024,        # samples per vmapped solve (bounds VRAM); one Adam step packs
                                  #   nRestarts*(1+2P) lanes so raise this on GPU to solve them in one pass
        "printEvery":  25,        # print the descent line every N Adam steps (0 = silent)
        # Optimizer parameter scaling. "linear": u = (theta-lo)/(hi-lo). "log": u = (log theta -
        # log lo)/(log hi - log lo), i.e. a fixed unit step is a fixed FRACTIONAL change in theta.
        # The calibration box is ~200x uneven under the linear map (the optimum sits at u*=0.003 for
        # C_Cs but u*=0.57 for E_Hl), so one lr step is 7% of E_Hl's own value and 780% of C_Cs's,
        # and the small-optimum compliances live one step from the lower wall. "log" equalises
        # relative resolution across all 16 (u* spread 218x -> 7x) and pulls them off the wall.
        "paramScale":  "log",     # "log" (equal relative resolution) | "linear" (raw box)

        # --- gd (batched Adam over the unit cube; all restarts stepped in LOCKSTEP) ----------
        "gd": {
            # All nRestarts descents advance TOGETHER: each Adam step evaluates the whole
            # nRestarts*(1+2P) finite-difference stencil in ONE vmapped batchedCalibration call, so
            # the forward solve is batched exactly like emcee/cmaes. (A per-point optimizer would be
            # ~nRestarts*maxIter serial un-vmapped solves -> multi-hour; this batching is the fix.)
            "gradSource": "fd",        # "fd" (central diff, robust default) | "autodiff" (vmapped, fragile)
            "nRestarts":  64,           # descents run in lockstep (cloud of optima for the corner / CV table)
            "initPool":   1024,         # LHS candidates screened by J; the nRestarts best finite seed the descents
            "lr":         0.04,        # Adam step size in unit space [0,1]^P
            "maxIter":    200,         # max Adam steps (batched -> each step is cheap)
            "gtol":       1e-4,        # per-restart early stop: ||grad||_2 below this freezes that restart
            "fdEps":      1e-3,        # central-diff step (FD gradient + Laplace Hessian), unit space
            "laplace":    True,        # add finite-diff Hessian cloud at the best optimum
            # Per-restart plateau decay. `metric` picks what "still improving" is measured on:
            #   "J"        - the summed objective (aggregate fit; the default). Cheap, smooth, and
            #                what the descent actually minimises.
            #   "worstRel" - max |rel err| over the observables. Cannot be starved by the residuals
            #                that already fit, but it is a max over noisy per-observable errors and so
            #                is a jumpier signal than J.
            # Independent of `metric`, a restart may only RETIRE once its worst |rel err| <= errTol;
            # one still above errTol keeps stepping and, after `stallLimit` floored plateaus, is frozen
            # and REPORTED as stalled (never silently counted as converged).
            "lrDecay": {
                "metric":     "J",     # "J" (aggregate objective) | "worstRel" (max |rel err|)
                "factor":     0.5,     # lr multiplier applied when a restart plateaus
                "patience":   50,      # window (Adam steps) over which improvement is measured
                "relTol":     1e-3,    # min fractional improvement of `metric` per window
                "minLr":      0.005,  # lr floor
                "errTol":     0.5,     # worst |rel err| (%) at/below which a floored restart may freeze
                "stallLimit": 3,       # consecutive floored plateaus above errTol -> freeze + report
            },
            # Adam moment decays / epsilon (rarely tuned; overridable here, .get fallbacks in the backend):
            # "beta1": 0.9, "beta2": 20.999, "adamEps": 1e-8,
        },
    },

    # --- output ------------------------------------------------------------
    "output": {
        "save": True,
        "path": "notebookData/convergence",
        "name": "mcmc_baseline_sepsis.h5",  # stem; setup appends the method (-> ..._gd.h5)
    },

    # --- paper artifacts (appendix table + figure; flip emit True for one run, then revert) ---
    "paper": {
        "emit":        False,     # True -> write the .tex table and the .png figure, then revert
        "dir":         "EFC_Paper/revision/generated",   # generated LaTeX fragments
        "imageDir":    "EFC_Paper/revision/Images",      # generated figures
        "table":       "gdSummary.tex",                  # non-float table -> Appendix (app:gd)
        "figure":      "gdTraces.png",                   # 6-panel search-trace figure
        # The emit cell is deliberately PURE-LOAD: it opens this file itself and depends on no
        # earlier cell, so the appendix artifacts can be regenerated without the setup cell above
        # (which still names the pre-rename cvModel.json / sepsis.json).
        "source":      "notebookData/convergence/mcmc_baseline_sepsis_gd.h5",
        # The four parameters that separate the two basins the restarts settle into. The SAME
        # four are drawn in mcmc_emcee.ipynb so the two appendix figures read side by side.
        "traceParams": ["R_As_Cs", "V0_As", "C_As", "E_Hl"],
        "scatterPair": ["R_As_Cs", "V0_As"],  # final-point scatter plane; the basins separate here
        "basinThresh": 2.0,          # worst |rel err| % splitting converged from trapped restarts
        # 1:1 at the manuscript's 8.5 cm column (3x2 grid): a wider figure downscaled into
        # the same column renders its 6-8 pt annotations unreadably small.
        "figSize":     [3.4, 5.2],   # inches, included at \includegraphics[width=8.5cm]
        "dpi":         200,
    },

    # --- analysis / plot ---------------------------------------------------
    "analysis": {
        "atm":             760.0,   # atmospheric offset for absolute-pressure signals
        "divergenceLimit": 1e6,     # |value| >= this in any obs/param = out of scope
        "errBand":         2.0,     # +/- tolerance band (%) drawn on the per-observation error boxplot
        "stepTraceYLim":   "initial",  # step-trace panel y-limits: "initial" (frame to iter-0 spread) | "robust" | None
        "stepTracePct":    [1, 99],    # robust percentile for the step-trace y-limit clip
    },

    # --- required by runner.buildSimulationParams (unused here) ------------
    "plots": [],
    "printStatus": False,

    # --- integration numerics (override scenario shared.integration; single config surface) ---
    "runTime": 10,       # simulated seconds per internal run
    "dt":      0.0005,   # integrator step
    "dtDense": 1.0,      # save/output grid: 1.0 = 1 Hz; raise for within-beat waveforms
}
# endregion

## Imports

<details>
<summary>Device/precision must be set from `runConfig` **before** `import jax`.</summary>



</details>

In [ ]:
# region -> imports + device/precision (must precede `import jax`)
# ---- repo-root bootstrap: run from any cwd (make `library` importable + resolve
# ---- the relative notebookData/ + config/ paths). Walks up to the dir containing library/. ----
import os, sys
_root = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(_root, "library")) and _root != os.path.dirname(_root):
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
os.chdir(_root)

useGpu    = runConfig["device"]["useGpu"]
precision = runConfig["device"]["precision"]

import os
if useGpu:
    os.environ.pop("CUDA_VISIBLE_DEVICES", None)
    os.environ["JAX_PLATFORMS"] = "cuda"
    os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
    os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"]   = "platform"
else:
    os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
    os.environ["JAX_PLATFORMS"] = "cpu"

import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", precision == "float64")

import numpy as np
from scipy.stats import qmc                      # Latin Hypercube sampling
import pandas as pd
import matplotlib.pyplot as plt
import corner                                    # posterior/identifiability marginals + pairwise
import h5py
import json
import time

import library.run.runner as runner              # buildSimulationParams
import library.run.stateSetup as stateSetup   # resolveCalibrationBounds (single-source calibration ranges)
import library.run.runnerBatchSI as runnerBatchSI  # batched (vmapped) forward solve
import library.model.modelClassSI as mc          # solveFinalArray / encodeStates (autodiff forward)
import library.run.progress as progressLib       # standard live-progress line + record log
import library.viz.plots as libPlots             # plotCalibrationConvergence (step traces)
import library.utils as utils
from library.hdf5 import schema_pop              # write_progress: persist the per-iteration progress log

print(f"jax {jax.__version__} | batched Adam GD | "
      f"x64={jax.config.jax_enable_x64} | devices={jax.devices()}")
# endregion

## Setup — `simulationParams`, prior bounds, targets

<details>
<summary>Cheap, scenario-only cells (no sampling). Re-run these + the Phase-2 load cell after a kernel</summary>

restart to re-plot a saved run. `buildForwardParams` swaps the scenario's staged calibration for
a single `multiplierC = 0.0` settle stage: the controller gain is clamped to zero so the 16
param-states stay frozen at their `Y0` values — a pure forward evaluation at θ.

</details>

In [ ]:
# region -> setup: simulationParams, prior bounds, forward evaluator
scenario = utils.loadScenario(runConfig["scenario"])
model    = utils.loadJSONfile(utils.configPath("models", runConfig["model"]))
conv         = scenario["convergence"]
calConf      = runner.buildSimulationParams(runConfig, scenario)["simulationConf"]["calibration"]
twin         = scenario["shared"]["twin"]["twinTargets"]
volDist      = scenario["shared"]["twin"]["volumeDistribution"]
param_names  = calConf["adaptive"]["parameters"]
observations = conv["observations"]
P            = len(param_names)
ic           = runConfig["inference"]
method       = ic["method"]
# output name carries the method so each calibration method's runs coexist on disk (..._gd.h5)
_stem, _ext  = os.path.splitext(runConfig["output"]["name"])
outPath      = os.path.join(runConfig["output"]["path"], f"{_stem}_{method}{_ext or '.h5'}")

# Prior bounds = the resolved calibration range (model min/max, overridable per scenario/runConfig):
# the hard limit EFC operates within and the single source of truth for the box — the correct prior.
# Same resolveCalibrationBounds the solve-time clamp and the convergence seeding box use.
bounds = stateSetup.resolveCalibrationBounds(model["calibration"], calConf, param_names)
lo, hi = bounds[:, 0], bounds[:, 1]

# Optimizer parameter scaling (runConfig["inference"]["paramScale"]): the box is mapped to the unit
# cube either linearly or in log space. Log makes a fixed unit step a fixed FRACTIONAL change in
# theta, so every parameter gets the same relative resolution regardless of its order of magnitude
# (the linear map leaves a ~200x spread in where the optimum sits inside its own box). All 16
# calibration parameters are strictly positive, so log is well defined; guarded below anyway.
paramScale = ic.get("paramScale", "linear")
if paramScale == "log" and np.any(lo <= 0):
    raise ValueError("paramScale='log' needs lo > 0 for every parameter; offending: "
                     f"{[param_names[i] for i in np.where(lo <= 0)[0]]}")
_fwdScale = np.log if paramScale == "log" else (lambda x: x)
_invScale = np.exp if paramScale == "log" else (lambda x: x)
tLo   = _fwdScale(lo)                       # unit-cube origin in the scaled coordinate
tSpan = _fwdScale(hi) - tLo                 # unit-cube span in the scaled coordinate


def buildForwardParams(settleRuns):
    """simulationParams whose calibration is one controllers-off (multiplierC=0) settle stage.
    buildSimulationParams returns `simulationConf` as the SAME scenario dict by reference, so we
    shallow-copy it and swap only `calibration` — else this clobbers the shared scenario."""
    sp = runner.buildSimulationParams(runConfig, scenario)
    conf = dict(sp["simulationConf"])
    conf["calibration"] = {
        "strategy": "staged",
        "maxCubicFactor": scenario["calibration"].get("maxCubicFactor", 10.0),
        "stages": [{
            "description": "GD baseline settle (controllers off)",
            "runsToIgnore": int(settleRuns), "runsToSave": 0,
            "multiplier": 0.0, "multiplierC": 0.0,
            "parameters": param_names, "targets": {}, "states": {},
        }],
    }
    sp["simulationConf"] = conf
    # silence the batched-solve progress line inside each forward eval (thousands of evals); the
    # live view is the per-restart line emitted by the backend loop (runGd).
    sp["solver"] = {**sp["solver"], "progressEvery": 0}
    return sp


def toUnit(theta):      # physical -> unit [0,1]^P (through the paramScale transform)
    return (_fwdScale(np.atleast_2d(np.asarray(theta, dtype=float))) - tLo) / tSpan

def toPhysical(u):      # unit [0,1]^P -> physical (through the paramScale transform)
    return _invScale(tLo + np.atleast_2d(np.asarray(u, dtype=float)) * tSpan)


sp   = buildForwardParams(ic["settleRuns"])
prep = runnerBatchSI.prepare(sp)                 # scalar setup once; reused every backend iteration

def forwardWith(sp_, prep_, theta):
    """theta (P,) or (n,P) -> full batchedCalibration result (rawObs (n,nObs) + finalStates)."""
    return runnerBatchSI.batchedCalibration(sp_, np.atleast_2d(np.asarray(theta, dtype=float)),
            param_names, observations, chunkSize=ic["chunkSize"], prepared=prep_, printStatus=False)

print(f"P = {P} params | observations = {len(observations)} | method = {method} | "
      f"prior = calibration bounds | scale = {paramScale} | settleRuns = {ic['settleRuns']} "
      f"(runTime = {sp['runTime']} s each)")
print(f"output -> {outPath}")
# endregion

In [ ]:
# region -> twin targets, atmospheric offsets, likelihood sigma
ATM = runConfig["analysis"]["atm"]

def obsOffset(name):
    """Atmospheric offset baked into absolute-pressure signals (gauge = raw - offset)."""
    return utils.obsOffset(name, ATM)

def obsTarget(name):
    """Twin target for an observation (gauge units), or None if untargeted. The targeted set is
    EFC's calibration set: the observables its cubic controllers drive (model['calibration']
    varTargets). Capillary means are a pressure DROP from the upstream arterial target, not
    absolute means. NOT targets (no controller drives them, so EFC never fits them):
      - Cyc_HC : cycle period = 60/HR, set by the driver (an INPUT).
      - V_Vs   : systemic venous volume -- the slack compartment absorbing the remaining blood
                 volume; no calibration controller targets it."""
    TBV = twin["TotalBloodVolume"]
    direct = {
        "avg_P_Vs": twin["CVP"],
        "avg_P_Cs": twin["Sys_P_As"] - twin["amp_P_As"] - twin["avg_P_Cs"],
        "avg_P_Cp": twin["Dia_P_Ap"] - twin["avg_P_Cp"],
        "keep_max_P_As": twin["Sys_P_As"], "keep_max_P_Ap": twin["Sys_P_Ap"],
        "keep_min_P_Ap": twin["Dia_P_Ap"], "amp_P_As": twin["amp_P_As"],
        "keep_SV_Hl": twin["CO"] / twin["HR"],
    }
    if name in direct:
        return direct[name]
    if name.startswith("avg_V_"):
        return volDist[name[len("avg_V_"):]] * TBV
    return None

targetArr = np.array([obsTarget(o) if obsTarget(o) is not None else np.nan for o in observations])
offsetArr = np.array([obsOffset(o) for o in observations])
targeted  = ~np.isnan(targetArr)                 # observations that have a twin target
tgt       = targetArr[targeted]                  # (nTargeted,) gauge-unit targets
sigma     = ic["sigmaRel"] * np.abs(tgt)         # single relative scale: sigma_i = sigmaRel*|t_i|
obsTargeted = [o for o, t in zip(observations, targeted) if t]
pd.DataFrame({"observation": obsTargeted, "target": tgt, "sigma": sigma})
# endregion

# Phase 1 — Run & save

<details>
<summary>Everything below samples the posterior and writes `outPath`. Skip to Phase 2 to re-plot a</summary>

previously saved file.

</details>

## Objective / likelihood / prior

<details>
<summary>Explicit objective (the definition reviewer #2 asked for):</summary>

$$J(\theta) = \sum_i \left(\frac{o_i(\theta) - t_i}{\sigma_i}\right)^2, \qquad \sigma_i = \varepsilon\,|t_i|.$$
Gaussian likelihood $\propto \exp(-\tfrac12 J)$; **uniform prior** over the safeguard bounds,
enforced by clipping each Adam step back into the unit cube. The optimizer works in **unit space**
$u\in[0,1]^P$ (θ = lo + u·span): the 16 params span ~0.02–450, and raw physical coordinates are
ill-conditioned for a first-order step. `log_prob` accepts an `(n, P)` block — one
`batchedCalibration` call scores the whole per-step gradient stencil in a single vmapped solve.

</details>

In [ ]:
# region -> log-posterior + prior (unit space) with in/out-of-bounds sanity checks
def log_prob(u):
    """Vectorized log-posterior in UNIT space u in [0,1]^P. u: (n,P) ->
    list of per-walker (logp, obsBlob): logp is the scalar log-posterior; obsBlob is the
    (nTargeted,) gauge-unit observation vector, captured by emcee as a blob so every step's
    observations are available for the step-trace plots (no recompute in Phase 2).
    Uniform prior (inside the box) x Gaussian likelihood exp(-J/2); diverged lanes (non-finite
    or astronomically large obs) get logp = -inf cleanly."""
    u = np.atleast_2d(np.asarray(u, dtype=float))
    # tol absorbs the whitening round-trip's ~1e-16 nudge at the [0,1] safeguard bound (emcee/SMC
    # sample in whitened coords, where whiteToUnit can push a boundary particle a hair out-of-box);
    # clip then scores it AT the bound instead of dropping it to -inf. Genuine out-of-box draws
    # (past tol) still fail the check -> logp = -inf below.
    inb = np.all((u >= -1e-9) & (u <= 1.0 + 1e-9), axis=1)
    u   = np.clip(u, 0.0, 1.0)
    obs = (forwardWith(sp, prep, toPhysical(u))["rawObs"] - offsetArr)[:, targeted]  # (n,nTargeted) gauge
    finite = np.all(np.isfinite(obs), axis=1) & np.all(np.abs(obs) < 1e12, axis=1)
    with np.errstate(over="ignore", invalid="ignore"):
        ll = -0.5 * np.sum(((obs - tgt) / sigma) ** 2, axis=1)
    good = inb & finite & np.isfinite(ll)
    ll = np.where(good, ll, -np.inf)
    return list(zip(ll, obs))                       # per-walker (logp, obs-blob)

def logp_only(theta):                               # scalar log-posterior for checks / physical theta
    return np.array([r[0] for r in log_prob(toUnit(np.atleast_2d(theta)))])

if runConfig["inference"].get("run", True):
    # sanity: in-bounds draws finite, out-of-bounds -inf
    _chk = qmc.scale(qmc.LatinHypercube(d=P, seed=123).random(4), lo, hi)
    print("log_prob(in-bounds draws):", np.round(logp_only(_chk), 2))
    print("log_prob(out-of-bounds)  :", np.array([r[0] for r in log_prob(np.full((1, P), 1.5))]))
# endregion

In [ ]:
# region -> preconditioning: warmup-covariance whitening + shared population init helper
if runConfig["inference"].get("run", True):
    # Whitening reparametrises the unit posterior by a WARMUP covariance so the curved target ridge
    # becomes ~isotropic in whitened coords w (u = uMean + w @ L.T, L = chol(cov)). The covariance is
    # learned self-contained: draw an LHS warmup batch, score it with log_prob (cell above), keep the
    # best-fitting points, whiten by their covariance. The affine-invariant stretch (emcee) and RW-MH
    # (SMC) assume roughly isotropic geometry, so this removes the migration / mixing cost along the
    # ridge. The map is AFFINE -> with a uniform box prior the target is unchanged up to a constant
    # Jacobian (no correction term). CMA-ES adapts its own covariance and works in raw unit coords, so
    # it IGNORES whitening -> its whiten block is disabled and the warmup below is skipped (no wasted
    # forward evals). The cell body is shared across the three notebooks.
    wc = ic.get("whiten", {})
    if wc.get("enabled", True):
        nWarm  = int(wc.get("warmup", 512))
        uWarm  = qmc.LatinHypercube(d=P, seed=ic["seed"]).random(nWarm)     # LHS warmup draws in unit box
        lpWarm = np.array([r[0] for r in log_prob(uWarm)])
        keep   = max(int(np.ceil(wc.get("keep", 0.2) * nWarm)), P + 1)      # top fraction by logp (floor P+1)
        top    = uWarm[np.argsort(lpWarm)[::-1][:keep]]
        uMean  = top.mean(0)
        cov    = np.cov(top.T) + 1e-9 * np.eye(P)             # ridge-jitter -> positive-definite Cholesky
        L      = np.linalg.cholesky(cov)
        Linv   = np.linalg.inv(L)
        def whiteToUnit(w): return uMean + np.atleast_2d(np.asarray(w, float)) @ L.T
        def unitToWhite(u): return (np.atleast_2d(np.asarray(u, float)) - uMean) @ Linv.T
        print(f"whitening ON | warmup {nWarm} -> kept {keep} | cov cond {np.linalg.cond(cov):.1f} | "
              f"stdev [{np.sqrt(np.diag(cov)).min():.3g}, {np.sqrt(np.diag(cov)).max():.3g}]")
    else:
        def whiteToUnit(w): return np.atleast_2d(np.asarray(w, float))
        def unitToWhite(u): return np.atleast_2d(np.asarray(u, float))
        print("whitening OFF | raw unit coords (CMA-ES ignores whitening; warmup skipped)")

    def logpW(w):
        """log_prob in whitened coords: w -> u -> the unit-space (logp, obsBlob) list (emcee / SMC reuse)."""
        return log_prob(whiteToUnit(w))

    def initUnit(n, rng):
        """n unit-space [0,1]^P init points via LHS (disperse over the prior box; self-contained)."""
        return qmc.LatinHypercube(d=P, seed=ic["seed"]).random(n)
else:
    print("plot-only: preconditioning/whitening skipped (only needed to seed the sampler)")
# endregion

In [ ]:
# region -> backend: batched gradient descent (lockstep Adam; FD/autodiff gradient + Laplace)
def _Junit(U):                                    # objective J = -2*logp over a (n,P) unit block
    return -2.0 * np.array([r[0] for r in log_prob(np.atleast_2d(U))], dtype=float)

def _batchValObs(U):
    """(n,P) unit -> (J (n,), obs (n,nTargeted) gauge) from ONE batched forward solve."""
    res = log_prob(np.atleast_2d(U))
    return -2.0 * np.array([r[0] for r in res]), np.array([r[1] for r in res])

def _worstRel(obs):
    """(n,nTargeted) gauge observations -> per-lane WORST |relative error| in %. This is the
    convergence metric the plateau detector gates on (see runGd): unlike the summed J it cannot be
    starved by the observables that already fit."""
    return 100.0 * np.max(np.abs(obs - tgt) / np.abs(tgt), axis=1)

def _fdHessianJ(x, eps=1e-3):
    """Central finite-difference Hessian of J at unit-space x (P,), assembled from ONE batched
    forward solve over the full ~(1+2P(P+1)) stencil. Returns (H, nEvalUsed)."""
    x = np.asarray(x, float); E = eps * np.eye(P); pts = [x]; idx = {}
    for i in range(P):
        for j in range(i, P):
            idx[(i, j)] = len(pts)
            pts += [np.clip(x+E[i]+E[j], 0, 1), np.clip(x-E[i]-E[j], 0, 1),
                    np.clip(x+E[i]-E[j], 0, 1), np.clip(x-E[i]+E[j], 0, 1)]
    J = _Junit(np.array(pts)); H = np.zeros((P, P))
    for (i, j), b in idx.items():
        H[i, j] = H[j, i] = (J[b] - J[b+2] - J[b+3] + J[b+1]) / (4 * eps * eps)
    return H, len(pts)

# ---- pure-jnp differentiable forward (OPTIONAL autodiff path; gradSource="autodiff") ------------
# Bypasses batchedCalibration's numpy layer: builds Y0 by scattering theta into the 16 param slots,
# chains `settleRuns` controllers-off solves via the existing solveFinalArray integrator, and gathers
# the SAME targeted observables the numpy path reads (states win, else algebraic outputs). jax.grad
# flows through solveFinalArray (pure jnp + lax.scan), BUT the model's valve/beat/systole jnp.where
# kinks nan-poison the reverse-mode gradient (dead branch inf at dispersed theta) -> unreliable. "fd"
# is the default; autodiff is kept (vmapped, guarded) for points where the kinks stay inactive.
_dtype    = jnp.zeros(0).dtype
_stage    = prep["stages"][0]                                       # single controllers-off stage
_cpModel  = _stage["cpModel"]
_constants = jnp.asarray(_stage["constantList"], dtype=_dtype)
_baseVec  = jnp.asarray(mc.encodeStates(prep["baseStates"], {"stateNames": prep["canonNames"]}))
_pIdx     = jnp.asarray([prep["nameIdx"][p] for p in param_names])  # 16 param slots in Y0
_outNames = list(_cpModel.outputNames); _outIdx = {n: i for i, n in enumerate(_outNames)}
_tLoJ, _tSpanJ = jnp.asarray(tLo), jnp.asarray(tSpan)      # scaled-coord box (see paramScale)
_invScaleJ     = jnp.exp if paramScale == "log" else (lambda x: x)
_offT     = jnp.asarray(offsetArr[targeted])                       # gauge offsets aligned to obsTargeted
_tgtJ, _sigJ = jnp.asarray(tgt), jnp.asarray(sigma)
_settle   = int(ic["settleRuns"])
_runTime, _dt0 = sp["runTime"], sp["dt"]
_solver   = {"type": sp["solver"].get("type", "euler")}
_gather = []                                                        # ("s",stateSlot) | ("o",outputSlot)
for o in obsTargeted:
    if o in prep["nameIdx"]:
        _gather.append(("s", prep["nameIdx"][o]))
    elif o in _outIdx:
        _gather.append(("o", _outIdx[o]))
    else:
        raise KeyError(f"targeted obs {o!r} is neither a state nor an algebraic output")

def _forwardJax(u):
    """unit-space u (P,) -> gauge targeted observations (nTargeted,), differentiable."""
    theta = _invScaleJ(_tLoJ + u * _tSpanJ)
    y = _baseVec.at[_pIdx].set(theta)
    finalOuts = jnp.zeros(len(_outNames), dtype=_dtype)
    for _ in range(_settle):                                        # chain settle runs (mirrors batchedCalibration)
        y, finalOuts = mc.solveFinalArray(_cpModel, _runTime, y, _constants, _dt0, 0.0, _solver)
    cols = [y[i] if kind == "s" else finalOuts[i] for kind, i in _gather]
    return jnp.stack(cols) - _offT

def _Jjax(u):                                     # autodiff objective; finite penalty on diverged lanes
    r = (_forwardJax(jnp.asarray(u)) - _tgtJ) / _sigJ
    r = jnp.where(jnp.isfinite(r), r, 1e3)
    return jnp.sum(r ** 2)

_gradRawV = jax.jit(jax.vmap(jax.grad(_Jjax)))    # batched autodiff gradient over a (n,P) block

def _batchGrad(U, gradSource, eps):
    """Gradient of J for a whole (n,P) unit block, batched. FD: central diff over the n*2P stencil
    (u +/- eps e_i) in ONE batched solve. autodiff: vmapped jax.grad (guarded via nan_to_num).
    Returns (grad (n,P), nLanesSolved)."""
    U = np.atleast_2d(np.asarray(U, float)); n = U.shape[0]
    if gradSource == "autodiff":
        g = np.nan_to_num(np.asarray(_gradRawV(jnp.asarray(U)), float), nan=0.0, posinf=0.0, neginf=0.0)
        return g, n
    E = eps * np.eye(P)
    plus  = np.clip(U[:, None, :] + E[None, :, :], 0, 1).reshape(n * P, P)
    minus = np.clip(U[:, None, :] - E[None, :, :], 0, 1).reshape(n * P, P)
    J = _Junit(np.vstack([plus, minus]))                           # one batched solve (chunked internally)
    grad = (J[:n * P].reshape(n, P) - J[n * P:].reshape(n, P)) / (2 * eps)
    return grad, 2 * n * P

def _screenStarts(gc, rng):
    """Feasibility-screened multi-start: draw an LHS pool, score with the guarded J, seed the
    descents from the nRestarts best FINITE / lowest-J candidates. Raw LHS over the full safeguard
    box lands many starts in the Euler-diverging region (J = +inf), where gradient descent cannot
    make progress; screening spends a cheap batched solve to start every descent feasible."""
    nPool = int(gc.get("initPool", max(8 * gc["nRestarts"], 64)))
    pool  = initUnit(nPool, rng)
    poolJ = _Junit(pool)
    order = np.argsort(np.where(np.isfinite(poolJ), poolJ, np.inf))
    nR    = int(gc["nRestarts"])
    starts = np.array([np.clip(pool[i], 0.0, 1.0) for i in order[:nR]])
    print(f"  init screen: {int(np.isfinite(poolJ).sum())}/{nPool} finite | seeding {nR} restarts "
          f"from best J [{poolJ[order[0]]:.3g} .. {poolJ[order[nR-1]]:.3g}]")
    return starts, nPool

def runGd():
    """Independent gradient-descent calibration: batched Adam minimising J = -2*logp on the unit
    cube [0,1]^P, all `nRestarts` feasibility-screened descents advanced in LOCKSTEP (one vmapped
    forward solve per step over the whole nRestarts*(1+2P) FD stencil). Box-clipped each step.
    Each restart carries its OWN lr with plateau decay measured on lrDecay.metric -- "J" (the summed
    objective, default) or "worstRel" (max |rel err|) -- over a `patience`-step window: no relTol
    fractional improvement -> lr *= factor, floored at minLr. Independent of the metric, a restart
    freezes only once it has plateaued AT the floor AND fits (worst |rel err| <= errTol), or after
    `stallLimit` floored plateaus while still above errTol (reported, not silent);
    ||grad|| < gtol likewise cannot freeze a restart that does not yet fit. chain = final optima + a Laplace Gaussian cloud
    (finite-diff Hessian of J at the best optimum -> covariance 2*inv(H)). Returns the common
    7-tuple in PHYSICAL coords; the shared step axis = the Adam iteration (each restart a 'walker')."""
    gc = ic["gd"]; rng = np.random.default_rng(ic["seed"])
    U, nPool = _screenStarts(gc, rng)                              # (nR, P) feasibility-screened
    nR = U.shape[0]
    lr    = gc.get("lr", 0.02); b1 = gc.get("beta1", 0.9); b2 = gc.get("beta2", 0.999)
    epsA  = gc.get("adamEps", 1e-8); gsrc = gc.get("gradSource", "fd"); fdEps = gc.get("fdEps", 1e-3)
    maxIter = int(gc["maxIter"]); gtol = gc.get("gtol", 1e-4)
    dc = gc.get("lrDecay", {})                                     # per-restart plateau decay of lr
    dFactor = dc.get("factor", 0.5); dPatience = int(dc.get("patience", 50))
    dRelTol = dc.get("relTol", 1e-3); dMinLr = dc.get("minLr", 1e-4)
    dErrTol = dc.get("errTol", 0.5); dStallLimit = int(dc.get("stallLimit", 3))
    dMetric = dc.get("metric", "J")                                # "J" | "worstRel"
    if dMetric not in ("J", "worstRel"):
        raise ValueError(f"lrDecay.metric must be 'J' or 'worstRel', got {dMetric!r}")
    def _progress(Jv, obsv):                                       # the quantity the plateau test watches
        return Jv if dMetric == "J" else _worstRel(obsv)
    lrVec = np.full(nR, float(lr))
    lastTest = np.zeros(nR, int); stalls = np.zeros(nR, int)
    m = np.zeros_like(U); v = np.zeros_like(U); active = np.ones(nR, bool)
    reporter = progressLib.ProgressReporter(logEnabled=runConfig["output"].get("logProgress", True))
    printEvery = max(1, ic.get("printEvery", 25) or 1)
    t0 = time.time(); nEval = nPool
    J, obs = _batchValObs(U); nEval += nR
    wBest = _worstRel(obs)                                          # best-so-far worst residual (%)
    mBest = _progress(J, obs); mHist = [mBest.copy()]               # best-so-far plateau metric
    traj, trajObs = [U.copy()], [obs.copy()]
    for it in range(1, maxIter + 1):
        g, nl = _batchGrad(U, gsrc, fdEps); nEval += nl
        g[~active] = 0.0                                           # frozen restarts take no step
        # gtol may retire a restart only if it actually fits -- a small gradient on a starved
        # coordinate is exactly the stall this run is meant to survive (stallLimit is the backstop).
        active &= (np.linalg.norm(g, axis=1) > gtol) | (wBest > dErrTol)
        if not active.any():
            break
        m = b1 * m + (1 - b1) * g                                 # Adam moments (bias-corrected)
        v = b2 * v + (1 - b2) * g * g
        step = lrVec[:, None] * (m / (1 - b1 ** it)) / (np.sqrt(v / (1 - b2 ** it)) + epsA)
        U = np.where(active[:, None], np.clip(U - step, 0.0, 1.0), U)
        J, obs = _batchValObs(U); nEval += nR
        wBest = np.minimum(wBest, _worstRel(obs))
        mBest = np.minimum(mBest, _progress(J, obs)); mHist.append(mBest.copy())
        # Plateau = lrDecay.metric failed to improve by relTol over the last `patience` steps.
        # Windowed (not per-step) so a slow-but-real descent is not misread as converged; re-tested
        # at most once per window per restart (lastTest).
        ref     = mHist[-1 - dPatience] if len(mHist) > dPatience else mBest
        plateau = (active & (it > dPatience) & ((it - lastTest) >= dPatience)
                   & ((ref - mBest) <= dRelTol * np.maximum(np.abs(ref), 1e-12)))
        lastTest = np.where(plateau, it, lastTest)
        decay    = plateau & (lrVec > dMinLr)
        lrVec    = np.where(decay, np.maximum(lrVec * dFactor, dMinLr), lrVec)
        atFloor  = plateau & ~decay                                # plateaued with lr already at minLr
        stalls   = np.where(atFloor, stalls + 1, np.where(decay, 0, stalls))
        fits     = wBest <= dErrTol
        freeze   = atFloor & (fits | (stalls >= dStallLimit))      # never freeze a still-descending misfit
        for r in np.where(freeze)[0]:                              # report, never silently truncate
            print(f"  restart {r} frozen at step {it}: worst |rel| {wBest[r]:.3f}% "
                  f"({'converged' if fits[r] else f'STALLED at lr floor x{stalls[r]}'})")
        active  &= ~freeze
        traj.append(U.copy()); trajObs.append(obs.copy())
        if it % printEvery == 0 or not active.any():
            reporter.emit(kind="step", label=(f"adam {it}/{maxIter} ({int(active.sum())} active, "
                          f"lr {lrVec.min():.2g}..{lrVec.max():.2g})"),
                          done=it, total=maxIter, elapsedWall=time.time() - t0,
                          stats=progressLib.relErrorStats(obs, tgt), acc=None)
        if not active.any():
            break

    optima = U; optVals = J
    chainSteps = np.stack([toPhysical(u) for u in traj])          # (nStep, nR, P)
    obsSteps   = np.stack(trajObs)                                # (nStep, nR, nTargeted)
    best = optima[int(np.argmin(optVals))]

    cloudU = optima.copy()
    if gc.get("laplace", True):
        H, nH = _fdHessianJ(best, fdEps); nEval += nH
        try:
            cov = 2.0 * np.linalg.inv(H); cov = 0.5 * (cov + cov.T)     # posterior exp(-J/2) -> 2*inv(H)
            draws  = rng.multivariate_normal(best, cov, size=max(200, 12 * P))
            cloudU = np.vstack([optima, np.clip(draws, 0.0, 1.0)])
        except np.linalg.LinAlgError:
            print("  Laplace skipped: Hessian not invertible")
    chain = toPhysical(cloudU)
    lp    = np.array([r[0] for r in log_prob(cloudU)]); nEval += cloudU.shape[0]
    fitWall = time.time() - t0
    diag = {"nRestarts": nR, "gradSource": gsrc, "initPool": nPool, "nSteps": len(traj) - 1,
            "laplace": bool(gc.get("laplace", True)), "bestJ": float(optVals.min()),
            "paramScale": paramScale, "plateauMetric": dMetric,
            "lrFinal": [float(x) for x in lrVec],
            "worstRelFinal": [float(x) for x in _worstRel(obs)],
            "nStalled": int((stalls >= dStallLimit).sum()),
            "accept": float("nan"), "progress": reporter.records}
    print(f"gd: {fitWall:.1f}s | {nR} restarts x {len(traj)-1} Adam steps | scale {paramScale} | "
          f"{int((stalls >= dStallLimit).sum())} stalled | best J {optVals.min():.4g} | "
          f"chain {chain.shape} (optima+Laplace) | {nEval} forward evals")
    return chain, lp, chainSteps, obsSteps, diag, fitWall, nEval
# endregion

## Run the selected backend

<details>
<summary>`runGd` descends the objective `J` over the controllers-off forward evaluator with batched Adam</summary>

— all `nRestarts` feasibility-screened descents advanced **in lockstep** on the unit cube, each
Adam step solving the whole `nRestarts × (1 + 2P)` gradient stencil in one vmapped
`batchedCalibration` call (box-clipped; per-restart early stop at `||grad|| < gtol`). The gradient
is central finite differences or autodiff (`inference.gd.gradSource`). It returns the same 7-tuple
`(chain, lp, chainSteps, obsSteps, diag, fitWall, nEval)` as the emcee/cmaes/smc siblings — the
step axis is the shared Adam iteration, each restart one descending trajectory — so the save and
Phase-2 plot cells never branch on method. **Runtime scales with Adam steps × restarts × (1 + 2P)
lanes, but batched into one solve per step — the full run is still a real experiment.**

</details>

In [ ]:
# region -> run the selected backend; collect the common tuple + shared convergence-error report
if runConfig["inference"].get("run", True):
    # The gd backend (runGd) descends the objective J over the controllers-off forward eval with
    # batched Adam (all restarts in lockstep) and returns the common 7-tuple, so Phase-2 save/plot
    # never branches on method.
    chain, lp, chainSteps, obsSteps, diag, fitWall, nEval = runGd()
    print(f"[{method}] chain {chain.shape} | steps {chainSteps.shape} | obs {obsSteps.shape} | "
          f"{nEval} forward evals | {fitWall:.1f}s")

    # progress trace for the persisted log: reuse the backend's live records if it kept them,
    # else reconstruct per-iteration convergence from obsSteps so every backend logs a comparable trace.
    if not diag.get("progress"):
        _pr = progressLib.ProgressReporter(logEnabled=runConfig["output"].get("logProgress", True))
        _every = max(1, ic.get("printEvery", 25) or 1); _nIt = obsSteps.shape[0]
        for _it in range(_nIt):
            if (_it + 1) % _every == 0 or _it == _nIt - 1:
                _pr.emit(kind="step", label=f"iter {_it + 1}/{_nIt}", done=_it + 1, total=_nIt,
                         elapsedWall=float("nan"), acc=diag.get("accept"),
                         stats=progressLib.relErrorStats(obsSteps[_it], tgt), show=False)
        diag["progress"] = _pr.records

    # --- calibration error vs twin targets (|rel err|; the backend analogue of a staged calibration):
    # --- watch the restarts collapse from their dispersed LHS start (Adam step 0) onto the targets
    # --- (final step), over all targets. nan-aware reducers for safety.
    _r    = np.abs(obsSteps - tgt) / np.abs(tgt)                   # (nIter, nRestart, nTargeted) relative error
    meanR = np.nanmean(_r, axis=2) * 100.0                        # per (iter, restart) mean|rel err| %
    maxR  = np.nanmax(_r, axis=2) * 100.0                         # per (iter, restart) max|rel err| %
    im, fm = np.nanmean(meanR[0]), np.nanmean(meanR[-1])          # ensemble-mean mean|rel|: step 0 -> final
    ix, fx = np.nanmean(maxR[0]),  np.nanmean(maxR[-1])           # ensemble-mean  max|rel|: step 0 -> final
    print(f"calibration error [all {len(tgt)} targets] (mean over restarts):")
    print(f"  mean|rel| {im:5.1f}% -> {fm:4.1f}% | max|rel| {ix:5.1f}% -> {fx:4.1f}% | "
          f"best restart max|rel| {np.nanmin(maxR):.2f}%")
else:
    print("plot-only: gradient-descent run skipped; Phase 2 loads the saved fit")
# endregion

## Save the posterior

<details>
<summary>Self-contained `.h5`: the flattened physical chain, log-prob, targets/sigma/bounds, and the</summary>

equivalence observables at MAP / mean (pre-computed so Phase 2 needs no forward solves).
`runConfig` + twin as attrs.

</details>

In [ ]:
# region -> save the fit + equivalence observables to a self-contained .h5
if runConfig["inference"].get("run", True):
    # equivalence observables (computed here so Phase 2 is pure-load): MAP/mean are fit outputs -> a
    # live forward solve at each.
    theta_MAP  = chain[np.argmax(lp)]
    theta_mean = chain.mean(axis=0)
    fitObs     = (forwardWith(sp, prep, np.array([theta_MAP, theta_mean]))["rawObs"] - offsetArr)[:, targeted]
    equivLabels = ["MAP", "mean"]
    equivObs    = fitObs                                              # (2, nTargeted) gauge

    if runConfig["output"]["save"]:
        os.makedirs(runConfig["output"]["path"], exist_ok=True)
        with h5py.File(outPath, "w") as f:
            f.create_dataset("chain", data=chain, compression="gzip")
            f.create_dataset("log_prob", data=lp, compression="gzip")
            # full per-iteration trajectories (nIter, nPop, .) for the step-trace plots
            f.create_dataset("chain_steps", data=chainSteps, compression="gzip")
            f.create_dataset("obs_steps", data=obsSteps, compression="gzip")
            f.create_dataset("targets", data=tgt)
            f.create_dataset("sigma", data=sigma)
            f.create_dataset("bounds", data=bounds)
            f.create_dataset("equiv_theta", data=np.array([theta_MAP, theta_mean]))
            f.create_dataset("equiv_obs", data=equivObs)
            f.create_dataset("equiv_labels", data=np.array(equivLabels, dtype="S"))
            f.create_dataset("param_names", data=np.array(param_names, dtype="S"))
            f.create_dataset("observation_names", data=np.array(obsTargeted, dtype="S"))
            f.attrs["runConfig"] = json.dumps(runConfig)
            f.attrs["twinTargets"] = json.dumps(twin)
            f.attrs["method"] = method
            f.attrs["diag"] = json.dumps(diag)                        # per-backend diagnostics (incl. progress log)
            f.attrs["accept"] = float(diag.get("accept", float("nan")))
            f.attrs["fitWall"] = fitWall
            f.attrs["nEval"] = int(nEval)
            f.attrs["settleRuns"] = ic["settleRuns"]
            f.attrs["runTime"] = sp["runTime"]
        schema_pop.write_progress(outPath, diag.get("progress"), meta={
            "model": runConfig["model"], "scenario": runConfig["scenario"], "method": method,
            "solver": sp["solver"]["type"], "nPop": int(obsSteps.shape[1]), "stack": "SI"})
        print(f"wrote {outPath}  (method={method}, chain {chain.shape}, steps {chainSteps.shape})")
else:
    print("plot-only: save skipped -- not overwriting the saved .h5")
# endregion

# Phase 2 — Load & analyse (from the saved file)

<details>
<summary>Everything below reconstructs from the `.h5` — no in-session sampler state required. After a</summary>

kernel restart, run the setup cells (config → imports → bounds/targets, all cheap) then this
load cell and the plot/table cells, without re-sampling.

</details>

In [ ]:
# region -> load everything the analysis needs, straight from the saved file
with h5py.File(outPath, "r") as f:
    chain       = f["chain"][:]
    lp          = f["log_prob"][:]
    chainSteps  = f["chain_steps"][:]
    obsSteps    = f["obs_steps"][:]
    tgt         = f["targets"][:]
    sigma       = f["sigma"][:]
    bounds      = f["bounds"][:]
    equivTheta  = f["equiv_theta"][:]
    equivObs    = f["equiv_obs"][:]
    equivLabels = list(f["equiv_labels"].asstr()[:])
    param_names = list(f["param_names"].asstr()[:])
    obsTargeted = list(f["observation_names"].asstr()[:])
    method      = f.attrs.get("method", "gd")
    diag        = json.loads(f.attrs.get("diag", "{}"))
    accept      = float(f.attrs.get("accept", float("nan")))
    fitWall     = f.attrs["fitWall"]
    nEval       = f.attrs["nEval"]
    settleRuns  = f.attrs["settleRuns"]
    runTime     = f.attrs["runTime"]
lo, hi = bounds[:, 0], bounds[:, 1]
print(f"loaded {outPath}: method={method} | chain {chain.shape} | steps {chainSteps.shape} | "
      f"{nEval} forward evals | accept {accept:.3f}")
# endregion

## Step traces — observations & parameters (batch-style)

<details>
<summary>The same per-signal panels as the convergence run, but the x-axis is the **Adam step** and each</summary>

jet line is one **restart**. Observations carry their twin **target** (dashed); parameters carry
the **prior bounds** (dotted). The screened LHS start is step 0, so you watch the restarts begin
dispersed across the prior and descend toward the target-consistent region — the gradient-descent
analogue of the calibration controllers driving each observation onto its target. All restarts
share the step axis (lockstep Adam); a restart that hits `gtol` early simply stops moving. The
spread that *remains* across restart optima is the identifiability width: tight ⇒ well-determined;
a persistent band ⇒ a sloppy / non-identifiable direction (the same manifold the corner plot shows).

</details>

In [ ]:
# region -> step traces: one panel per targeted observation (population vs target)
# --- step traces: one panel per targeted observation (each restart = a jet line vs its target) ---
# obsSteps is (nStep, nRestart, nTargeted) gauge-unit observations per Adam step (all restarts share
# the step axis / lockstep; a restart that hits gtol just stops moving). Per panel: every restart's
# observation value vs step (jet), with its twin target (dashed). The fit analogue of the
# calibration-convergence plot: the restarts start dispersed and descend onto each target; the
# residual spread across optima is the identifiability width. `model` (setup cell) maps obs -> controller.
obsTraces    = {o: obsSteps[:, :, k].T for k, o in enumerate(obsTargeted)}      # {obs:(nRestart,nStep)}
targetsByObs = {o: float(t) for o, t in zip(obsTargeted, tgt)}
paramForObs  = {model["calibration"][p]["params"]["varTarget"]: p
                for p in param_names if p in model.get("calibration", {})}

libPlots.plotCalibrationConvergence(
    obsTraces, traceT=None, targets=targetsByObs, paramForObs=paramForObs, showLegend=False,
    ylim=runConfig["analysis"]["stepTraceYLim"], robustPct=tuple(runConfig["analysis"]["stepTracePct"]),
    divLimit=runConfig["analysis"]["divergenceLimit"],
    title=f"{method} step traces -- observations (restarts, all Adam steps)")
plt.show()
# endregion

In [ ]:
# region -> step traces: one panel per swept parameter (population)
# --- step traces: one panel per swept parameter (each restart = a jet line) ------------------
# Each panel: every restart's parameter value vs Adam step (jet), with the prior bounds (dotted
# grey). No target line -- parameters have no fixed target; the spread that remains across the
# restart optima at the final step is that parameter's identifiability.
paramTraces   = {p: chainSteps[:, :, j].T for j, p in enumerate(param_names)}   # {param:(nRestart,nStep)}
rangesByParam = {p: (float(lo[j]), float(hi[j])) for j, p in enumerate(param_names)}

libPlots.plotCalibrationConvergence(
    paramTraces, traceT=None, ranges=rangesByParam, paramForObs=None, showLegend=False,
    ylim=runConfig["analysis"]["stepTraceYLim"], robustPct=tuple(runConfig["analysis"]["stepTracePct"]),
    divLimit=runConfig["analysis"]["divergenceLimit"],
    title=f"{method} step traces -- parameters (restarts, all Adam steps)")
plt.show()
# endregion

## Per-observation error boxplot

In [ ]:
# region -> boxplot: relative error distribution per observation (converged population)
# --- per-observation relative error across the CONVERGED restart optima (final Adam step): the fit
# analogue of the batch/serial convergence-run boxplot, each restart optimum playing the role of a
# "run". Dashed red lines mark the +/- errBand tolerance band. All restarts share the final step
# (lockstep); each observation's column is filtered to its finite members for safety.
errTarget = runConfig["analysis"].get("errBand", 2.0)
finalPop  = obsSteps[-1]                                         # (nRestart, nTargeted) gauge obs, optima
errRel    = (finalPop - tgt) / tgt * 100.0                      # (nRestart, nTargeted) relative error %
cols      = [errRel[np.isfinite(errRel[:, k]), k] for k in range(errRel.shape[1])]
fig, ax = plt.subplots(figsize=(12, 5))
ax.boxplot(cols, tick_labels=utils.labelsFor(obsTargeted, "latex"), showfliers=False)
ax.axhline(0.0, color="k", lw=0.8)
ax.axhline(errTarget, color="r", ls="--", lw=0.8, label=f"±{errTarget}% target")
ax.axhline(-errTarget, color="r", ls="--", lw=0.8)
ax.set_ylabel("relative error (%)")
ax.set_title(f"{method} convergence error across {finalPop.shape[0]} restart optima (final Adam step)")
ax.tick_params(axis="x", rotation=90)
ax.legend()
plt.tight_layout()
plt.show()
# endregion

## Equivalence + cost

<details>
<summary>1. **Equivalence** — does the best optimum (MAP) hit the targets? (from stored obs.)</summary>

2. **Cost** — the total short forward solves for the fit (`nRestarts` descents × iterations ×
per-gradient solves + the Laplace Hessian stencil).

</details>

In [ ]:
# region -> equivalence table: MAP / mean fit relative error per observable
errCmp = (equivObs - tgt) / tgt * 100.0                 # (K, nTargeted) relative error %
eqTable = pd.DataFrame({"observation": obsTargeted, "target": tgt})
for i, name in enumerate(equivLabels):
    eqTable[f"{name}_relerr_%"] = errCmp[i]
print("max |rel err| %:", {n: float(np.nanmax(np.abs(errCmp[i]))) for i, n in enumerate(equivLabels)})
eqTable
# endregion

In [ ]:
# region -> cost: fit forward evals + wall time
print(f"{method}: {nEval} forward evals ({settleRuns} x {runTime} s each) -> {fitWall:.1f} s wall "
      f"| accept {accept:.3f} | diag {diag}")
# endregion

## Paper artifacts — appendix summary table + search-trace figure


<details>

The two artifacts for the manuscript's gradient-descent-baseline appendix subsection
(`app:gd`). The table mirrors the shape of the EFC convergence table
(`tab:summaryTable`) so the baseline reads against it row for row, with statistics taken
over **all** restart optima — the large standard deviations are the finding, not noise:
the restarts do not agree with one another.

The figure is the paper-sized counterpart of the full step-trace grid above. Restarts are
coloured by outcome (`worst |rel err| <= runConfig["paper"]["basinThresh"]`), and the
four parameters drawn are the ones on which the two basins separate; the final-point
scatter shows them as two disjoint clusters.

**This cell is self-contained** — it opens `runConfig["paper"]["source"]` itself and
needs nothing from the setup cell, which still names the pre-rename `cvModel.json` /
`sepsis.json`. Run the runConfig + imports cells, then this one.

Both files are written only when `runConfig["paper"]["emit"]` is `True`; flip it for one
run and revert.

</details>


In [ ]:
# region -> paper artifacts: appendix summary table + compact search-trace figure (pure-load)
# Self-contained: reads the saved .h5 directly, so it needs only the runConfig + imports cells.
# Population for every statistic = ALL restart optima at the final Adam step. That is the honest
# mirror of tab:summaryTable ("across runs"), and it is why the std columns are large: the
# restarts disagree. pObs[k] / pParam[k] are the controller PAIR (observable k is the target of
# parameter k), which is what lets each table row carry both halves as tab:summaryTable does.
pc = runConfig["paper"]
with h5py.File(pc["source"], "r") as f:
    pSteps  = f["chain_steps"][:]                    # (nStep, nRestart, nParam) physical
    pObsSt  = f["obs_steps"][:]                      # (nStep, nRestart, nTargeted) gauge
    pTgt    = f["targets"][:]
    pBounds = f["bounds"][:]
    pParam  = list(f["param_names"].asstr()[:])
    pObs    = list(f["observation_names"].asstr()[:])
    pDiag   = json.loads(f.attrs.get("diag", "{}"))

pFinalObs = pObsSt[-1]                                              # (nRestart, nTargeted)
pWorst    = 100.0 * np.max(np.abs(pFinalObs - pTgt) / np.abs(pTgt), axis=1)   # per-restart worst %
pGood     = pWorst <= pc["basinThresh"]
pOptima   = pSteps[-1]                                              # (nRestart, nParam)
pErrRel   = (pFinalObs - pTgt) / pTgt * 100.0                       # signed relative error, %

fig = libPlots.plotSearchTraces(
    pSteps, pObsSt, pTgt, pBounds, pParam,
    pc["traceParams"], scatterPair=pc["scatterPair"], groupMask=pGood,
    errBand=pc["basinThresh"], figSize=tuple(pc["figSize"]))
plt.show()

# the basin the trapped restarts share, named by the parameter that separates them most
_ratio  = np.array([max(a, b) / max(min(a, b), 1e-12) for a, b in
                    zip(np.median(pOptima[pGood], axis=0), np.median(pOptima[~pGood], axis=0))])
_sepP   = pParam[int(np.argmax(_ratio))]
_sepGood, _sepBad = (float(np.median(pOptima[pGood, int(np.argmax(_ratio))])),
                     float(np.median(pOptima[~pGood, int(np.argmax(_ratio))])))
_badLo, _badHi = float(pWorst[~pGood].min()), float(pWorst[~pGood].max())

rows = {}
for k, (o, p) in enumerate(zip(pObs, pParam)):
    rows[utils.labelFor(o, "latex")] = [
        f"{pTgt[k]:.4g}", f"{pErrRel[:, k].mean():.3f}", f"{pErrRel[:, k].std():.3f}",
        utils.labelFor(p, "latex"), f"{pOptima[:, k].mean():.3f}", f"{pOptima[:, k].std():.3f}"]

latex = utils.generateLatexTableInline(
    rows, ["Vars.", "Targets", "Relative Error \\%", "std", "Param.", "Value", "std"],
    ref="tab:gdSummary",
    colSpec="C{0.7cm} C{0.7cm} |C{1.0cm} C{0.7cm}|C{0.7cm} C{0.7cm} C{0.7cm}",
    fontSize="small",
    caption=(
        "Multi-start gradient-descent baseline calibration, reported in the same form as the "
        "Embedded Feedback Controller convergence test of Table~\\ref{tab:summaryTable}: the same "
        "sixteen model parameters driven to the same sixteen physiological targets, over the same "
        "prior box and under the same explicit-Euler integration. Statistics are taken over all "
        f"{pOptima.shape[0]} restart optima. For each calibration target the table reports the "
        "prescribed target value, the mean signed relative error and its standard deviation over "
        "the restarts, together with the corresponding calibrated parameter's mean and its "
        "variability across restarts. The standard deviations are large because the restarts do "
        f"not agree: {int(pGood.sum())} of the {pOptima.shape[0]} reach every target to within "
        f"{pc['basinThresh']:g}\\,\\%, while the remaining {int((~pGood).sum())} settle in a "
        f"distinct spurious basin at {utils.labelFor(_sepP, 'latex')} $\\approx$ {_sepBad:.3g} "
        f"instead of {_sepGood:.3g}, where the worst target is missed by {_badLo:.0f}--"
        f"{_badHi:.0f}\\,\\%. Nothing in the optimiser's own output distinguishes the two "
        "groups; their histories are shown in Figure~\\ref{fig:gdTraces}."))

print(f"{int(pGood.sum())}/{pOptima.shape[0]} restarts within {pc['basinThresh']:g}% | "
      f"trapped basin {_sepP} {_sepBad:.3g} vs {_sepGood:.3g} | trapped worst |rel| "
      f"{_badLo:.1f}-{_badHi:.1f}% | diag nStalled={pDiag.get('nStalled')}")

if pc["emit"]:
    os.makedirs(pc["dir"], exist_ok=True)
    os.makedirs(pc["imageDir"], exist_ok=True)
    _t = os.path.join(pc["dir"], pc["table"])
    with open(_t, "w") as fh:
        fh.write(latex)
    print(f"wrote {_t}")
    _f = os.path.join(pc["imageDir"], pc["figure"])
    fig.savefig(_f, dpi=pc["dpi"], bbox_inches="tight")
    print(f"wrote {_f}")
else:
    print("paper.emit is False -- table/figure not written")
# endregion


In [ ]:
# region -> release GPU memory
import gc
for _v in ("chain", "chainSteps", "obsSteps", "lp", "prep", "equivObs"):
    globals().pop(_v, None)
jax.clear_caches()
gc.collect()
try:
    used = sum((d.memory_stats() or {}).get("bytes_in_use", 0) for d in jax.devices())
    print(f"GPU bytes in use after cleanup: {used/1e6:.0f} MB (restart kernel for a full reset)")
except Exception as e:
    print("memory_stats() unavailable on this device:", e)
# endregion